# Task 2: Imitation Learning

## Setup Code / Packages
If you are running the code locally follow the instructions [here](../docs/setup.md).

If you are using Google Colab, run the two cells below. The first clones the
repository and installs dependencies; the second selects the rendering backend.

> **Set `REPO_URL` to your own fork first.** The Panda MJCF fixes this task
> depends on live in `assets/descriptions/panda/mjcf/`, so cloning upstream or
> uploading only your scene XML will leave the cube ungraspable.


In [12]:
# --- Colab bootstrap ----------------------------------------------------
# No-op when running locally. In Colab this clones the repo and installs deps.
import importlib.util, os, subprocess, sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# POINT THIS AT YOUR OWN FORK before running in Colab.
# The task 2 fixes to the Panda MJCF - collision flags and gripper stiffness in
# assets/descriptions/panda/mjcf/ - are what make the cube graspable at all.
# Uploading just DropCubeInBinEnv.xml does NOT bring them along, and without
# them the gripper passes through the cube and every episode fails.
REPO_URL = "https://github.com/triton-droids/simulation.git"
BRANCH = "ml_onboarding"

if IN_COLAB:
    # mplib publishes Linux wheels for cp38-cp312 only. On cp313+ the install
    # falls back to a source build and fails.
    print("Python", sys.version.split()[0])

    if not os.path.isdir("simulation"):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("/content/simulation")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "mujoco==3.3.4", "gymnasium[mujoco]==0.29.1", "mplib", "moviepy", "imageio-ffmpeg"],
        check=True,
    )
    print("working directory:", os.getcwd())


In [13]:
import os, sys

# MuJoCo rendering backend.
#   macOS  -> glfw
#   Colab  -> egl, which renders offscreen with no display server. The previous
#             "glfw on Linux" setting needs an X display and fails on a headless
#             Colab VM unless you also start Xvfb.
if sys.platform == "darwin":
    os.environ["MUJOCO_GL"] = "glfw"
else:
    os.environ["MUJOCO_GL"] = "egl"

print("MUJOCO_GL =", os.environ["MUJOCO_GL"])


## Using Gymnasium
No problems are assigned here, but understanding the code APIs/interfaces and functionalities are important.

### Gymnasium Basics
The Gymnasium package is a standardized API for creating and interacting with simulation environments. 

<img src="../assets/media/gym_loop.png" alt="agent-environment loop" width="500"/>

Gymnasium defines the agent-environment loop — a simple interface where an agent takes actions and receives observations, rewards, and termination signals from an environment. This loop forms the foundation for reinforcement learning (RL) and allows us to easily manipulate the simulator.

**Why We Use Gymnasium**

Gymnasium acts as a common interface layer between learning algorithms and simulation environments. It provides:
- A uniform API (reset(), step(), observation, reward, done, info) 
- Integration with RL algorithm libraries (e.g., Stable-Baselines3, CleanRL).
- Rendering and debugging utilities for visualization and evaluation.


In this task, you will use Gymnasium as the primary API to interface with the simulator to :
- Collect trajectory data for imitation learning.
- Evaluate and deploy trained policies.

Read their documentation [here](https://gymnasium.farama.org/).


## Part 0: Creating a Standardized Environment with Gymnasium
In this part, you will implement the `TrajEnv` class to wrap a MuJoCo simulation in a Gymnasium interface.

Your task is to decide which observations are useful as states in a behavior cloning setting. You will also implement the `is_grasped` property to define when the robot has successfully grasped the object and the `terminated` property to specify when the cube is in the bin. The `reset_model` method should randomize initial arm and object positions to encourage generalization.

You should step through Gymnasium functions like `do_simulation` and `set_state` to see how to control the simulator.

In [15]:
# NOTE: reference scaffold only. The next cell redefines TrajEnv with the
# task logic filled in - that is the version to run.
from gymnasium.envs.mujoco import MujocoEnv
from gymnasium.spaces import Box
import numpy as np
import mujoco
import os

XML_PATH = os.path.abspath("assets/descriptions/DropCubeInBinEnv.xml")
assert os.path.exists(XML_PATH), f"Missing XML at {XML_PATH}"


def require_id(model, obj_type, name):
    """Look up a named element, failing loudly if it is missing.

    mujoco.mj_name2id returns -1 for an unknown name rather than raising, and -1
    silently indexes the LAST row of data.xpos / data.site_xpos. A typo therefore
    shows up much later as a mysteriously wrong pose instead of an error here.
    """
    idx = mujoco.mj_name2id(model, obj_type, name)
    if idx < 0:
        raise ValueError(f"No {mujoco.mjtObj(obj_type).name} named {name!r} in this model.")
    return idx


class TrajEnv(MujocoEnv):
    metadata = {"render_modes": ["human", "rgb_array", "depth_array"]}

    def __init__(self, xml_file: str, frame_skip: int = 5, **kwargs):
        # Build a temporary model to size the observation BEFORE calling super().__init__
        tmp_model = mujoco.MjModel.from_xml_path(xml_file)
        obs_dim = tmp_model.nq + tmp_model.nv
        if obs_dim == 0:
            print("Warning: observation dimension is 0. Complete Task 1 scene setup first.")

        observation_space = Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

        super().__init__(
            xml_file,
            frame_skip,
            observation_space=observation_space,
            **kwargs,
        )

        # Names as they appear in assets/descriptions/DropCubeInBinEnv.xml.
        # "link0" is the arm base; it is the MJCF counterpart of the URDF's
        # "panda_link0", which is the frame the motion planner plans relative to.
        self.base_id = require_id(self.model, mujoco.mjtObj.mjOBJ_BODY, "link0")
        self.cube_id = require_id(self.model, mujoco.mjtObj.mjOBJ_BODY, "cube")

    @property
    def is_grasped(self) -> bool:
        # TODO: implement using contacts / distances. Stub is False so it runs.
        return False

    @property
    def terminated(self) -> bool:
        # TODO: success condition, e.g., cube in bin.
        return False

    def _get_obs(self) -> np.ndarray:
        # Generic observation = [qpos, qvel]
        qpos = self.data.qpos if self.data.qpos is not None else np.zeros(self.model.nq)
        qvel = self.data.qvel if self.data.qvel is not None else np.zeros(self.model.nv)
        obs = np.concatenate([qpos.ravel(), qvel.ravel()]).astype(np.float32)
        return obs

    def step(self, action: np.ndarray):
        # Gymnasium step must return: obs, reward, terminated, truncated, info
        self.do_simulation(action, self.frame_skip)
        obs = self._get_obs()
        reward = 0.0            # TODO: shape a task reward
        terminated = self.terminated
        truncated = False       # TimeLimit wrapper should handle truncation
        info = {}
        if self.render_mode == "human":
            self.render()
        return obs, reward, terminated, truncated, info

    def reset_model(self):
        # Reset to keyframe 'home' if present, else zeros
        try:
            home = self.model.keyframe("home")
            default_qpos = np.array(home.qpos, dtype=np.float32)
        except Exception:
            default_qpos = np.zeros(self.model.nq, dtype=np.float32)

        qpos = default_qpos.copy()
        qvel = np.zeros(self.model.nv, dtype=np.float32)

        # TODO: randomize initial arm/object states here
        self.set_state(qpos, qvel)
        return self._get_obs()


AssertionError: Missing XML at /Users/darin/Desktop/_/club_stuff/simulation/task2/assets/descriptions/DropCubeInBinEnv.xml

In [ ]:
from scipy.spatial.transform import Rotation as R
from gymnasium.envs.mujoco import MujocoEnv
from gymnasium.spaces import Box
import numpy as np
import mujoco
import os

XML_PATH = os.path.abspath("assets/descriptions/DropCubeInBinEnv.xml")


def require_id(model: mujoco.MjModel, obj_type, name: str) -> int:
    """Look up a named element, failing loudly if it is missing.

    mujoco.mj_name2id returns -1 for an unknown name rather than raising, and -1
    silently indexes the LAST row of data.xpos / data.site_xpos. A typo therefore
    shows up much later as a mysteriously wrong pose instead of an error here.
    """
    idx = mujoco.mj_name2id(model, obj_type, name)
    if idx < 0:
        raise ValueError(f"No {mujoco.mjtObj(obj_type).name} named {name!r} in this model.")
    return idx


class TrajEnv(MujocoEnv):
    """Gymnasium wrapper around the DropCubeInBin scene.

    Shapes for this scene:
      observation = [qpos, qvel] = 16 + 15 = 31
      action      = 7 arm joint position targets (rad) + 1 gripper command (0-255)

    Args:
        cube_xy_noise: half-width of the uniform square the cube is spawned in,
            in metres, around its keyframe position. Start at 0.0 and only raise
            it once the planner succeeds reliably on the fixed layout.
    """

    metadata = {"render_modes": ["human", "rgb_array", "depth_array"]}

    # Bin geometry, read off bin/bin_body.xml and measured from the bin_center
    # site (which sits 0.03 above the bin body origin):
    #   walls at +-0.05 with half-thickness 0.005 -> inner faces at +-0.045
    #   floor top is 0.005 above the origin       -> 0.025 BELOW the site
    #   walls are 0.06 tall, rim top at 0.055     -> 0.025 ABOVE the site
    BIN_INNER_HALF_WIDTH = 0.045
    BIN_FLOOR_BELOW_SITE = 0.025
    BIN_RIM_ABOVE_SITE = 0.025
    SETTLED_SPEED = 0.1  # m/s; the cube must have come to rest, not be mid-bounce

    def __init__(self, xml_file: str, frame_skip: int = 5, cube_xy_noise: float = 0.0, **kwargs):
        # Starter observation: [qpos, qvel]. Extend this once your pipeline is stable.
        tmp_model = mujoco.MjModel.from_xml_path(xml_file)
        obs_dim = tmp_model.nq + tmp_model.nv
        if obs_dim == 0:
            print("Warning: observation dimension is 0. Complete Task 1 scene setup first.")

        observation_space = Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim,),
            dtype=np.float32,
        )

        super().__init__(
            xml_file,
            frame_skip,
            observation_space=observation_space,
            **kwargs,
        )

        self.cube_xy_noise = cube_xy_noise

        # --- Named elements, exactly as they appear in the MJCF ---
        # Bodies. "link0" is the arm base: the MJCF counterpart of the URDF's
        # "panda_link0", which is the frame the motion planner plans relative to.
        self.base_id = require_id(self.model, mujoco.mjtObj.mjOBJ_BODY, "link0")
        self.cube_id = require_id(self.model, mujoco.mjtObj.mjOBJ_BODY, "cube")
        self.left_finger_id = require_id(self.model, mujoco.mjtObj.mjOBJ_BODY, "left_finger")
        self.right_finger_id = require_id(self.model, mujoco.mjtObj.mjOBJ_BODY, "right_finger")

        # Sites. These are SITES, not bodies, so they live in data.site_xpos.
        self.tcp_sid = require_id(self.model, mujoco.mjtObj.mjOBJ_SITE, "panda_hand_tcp")
        self.bin_sid = require_id(self.model, mujoco.mjtObj.mjOBJ_SITE, "bin_center")

        # The cube's free joint: 7 qpos entries [x, y, z, qw, qx, qy, qz] starting
        # at jnt_qposadr, and 6 qvel entries [vx, vy, vz, wx, wy, wz] at jnt_dofadr.
        cube_joint_id = require_id(self.model, mujoco.mjtObj.mjOBJ_JOINT, "cube_free")
        self.cube_qpos_adr = int(self.model.jnt_qposadr[cube_joint_id])
        self.cube_dof_adr = int(self.model.jnt_dofadr[cube_joint_id])

        cube_geom_id = require_id(self.model, mujoco.mjtObj.mjOBJ_GEOM, "cube_geom")
        self.cube_half_size = float(self.model.geom_size[cube_geom_id][0])

    # --- Pose accessors -------------------------------------------------
    # All of these are in the WORLD frame, which is what the planner expects.
    # Quaternions follow MuJoCo's [w, x, y, z] convention.

    @property
    def base_pose(self) -> np.ndarray:
        """(7,) world pose of the arm base, for planner.set_base_pose."""
        return np.concatenate([self.data.xpos[self.base_id], self.data.xquat[self.base_id]])

    @property
    def tcp_pos(self) -> np.ndarray:
        """(3,) world position of the grasp point between the fingers."""
        return self.data.site_xpos[self.tcp_sid].copy()

    @property
    def tcp_quat(self) -> np.ndarray:
        """(4,) world orientation of the TCP as [w, x, y, z]."""
        quat = np.empty(4)
        mujoco.mju_mat2Quat(quat, self.data.site_xmat[self.tcp_sid])
        return quat

    @property
    def tcp_pose(self) -> np.ndarray:
        """(7,) world pose of the TCP as [x, y, z, qw, qx, qy, qz]."""
        return np.concatenate([self.tcp_pos, self.tcp_quat])

    @property
    def cube_pos(self) -> np.ndarray:
        return self.data.xpos[self.cube_id].copy()

    @property
    def cube_quat(self) -> np.ndarray:
        return self.data.xquat[self.cube_id].copy()

    @property
    def cube_linear_velocity(self) -> np.ndarray:
        """(3,) world linear velocity of the cube."""
        return self.data.qvel[self.cube_dof_adr : self.cube_dof_adr + 3].copy()

    @property
    def bin_pos(self) -> np.ndarray:
        """(3,) world position of the middle of the bin opening."""
        return self.data.site_xpos[self.bin_sid].copy()

    # --- Task logic -----------------------------------------------------

    def _touching(self, body_a: int, body_b: int) -> bool:
        """True if any active contact pairs geoms belonging to these two bodies."""
        for i in range(self.data.ncon):
            contact = self.data.contact[i]
            bodies = {
                int(self.model.geom_bodyid[contact.geom1]),
                int(self.model.geom_bodyid[contact.geom2]),
            }
            if bodies == {body_a, body_b}:
                return True
        return False

    @property
    def is_grasped(self) -> bool:
        """True when both fingertips are in contact with the cube.

        The fingertip pad geoms are unnamed in panda_body.xml, so contacts are
        matched by BODY: each contact stores two geom ids, and model.geom_bodyid
        maps a geom back to the body that owns it.

        Note this is also true for a moment while the gripper is closing around a
        cube that is still resting on the table. If you need "lifted and held",
        add: and self.cube_pos[2] > table_height + self.cube_half_size + 0.01
        """
        return (
            self._touching(self.cube_id, self.left_finger_id)
            and self._touching(self.cube_id, self.right_finger_id)
        )

    @property
    def terminated(self) -> bool:
        """True when the cube has come to rest inside the bin.

        Three conditions, all measured relative to the bin_center site:
          1. the cube's centre is inside the bin's inner walls in XY
          2. it is above the bin floor and below the rim, i.e. actually dropped in
             rather than resting on top of a wall
          3. it has stopped moving, so a cube still falling does not count early
        """
        offset = self.cube_pos - self.bin_pos

        inside_xy = np.all(np.abs(offset[:2]) < self.BIN_INNER_HALF_WIDTH)
        below_rim = -self.BIN_FLOOR_BELOW_SITE < offset[2] < self.BIN_RIM_ABOVE_SITE
        settled = np.linalg.norm(self.cube_linear_velocity) < self.SETTLED_SPEED

        return bool(inside_xy and below_rim and settled)

    def _get_obs(self):
        # TODO: Extend with task-relevant features once baseline is running.
        qpos = self.data.qpos if self.data.qpos is not None else np.zeros(self.model.nq)
        qvel = self.data.qvel if self.data.qvel is not None else np.zeros(self.model.nv)
        return np.concatenate([qpos.ravel(), qvel.ravel()]).astype(np.float32)

    def step(self, action: np.ndarray) -> tuple[np.ndarray, float, bool, bool, dict]:
        self.do_simulation(action, self.frame_skip)
        obs = self._get_obs()
        reward = 0.0  # TODO: optional shaping reward for diagnostics.
        terminated = self.terminated

        if self.render_mode == "human":
            self.render()

        # truncation=False as the TimeLimit wrapper handles episode truncation.
        # Always wrap this env in TimeLimit, or a policy that never succeeds will
        # loop forever in eval_policy.
        return obs, reward, terminated, False, {"is_grasped": self.is_grasped}

    def reset_model(self):
        try:
            default_qpos = np.array(self.model.keyframe("home").qpos, dtype=np.float64)
        except Exception:
            default_qpos = np.zeros(self.model.nq, dtype=np.float64)

        qpos = default_qpos.copy()
        qvel = np.zeros(self.model.nv, dtype=np.float64)

        # Randomise only the cube's XY, and only if asked. self.np_random is the
        # Gymnasium RNG seeded by reset(seed=...), so episodes stay reproducible.
        if self.cube_xy_noise > 0.0:
            qpos[self.cube_qpos_adr : self.cube_qpos_adr + 2] += self.np_random.uniform(
                low=-self.cube_xy_noise, high=self.cube_xy_noise, size=2
            )

        self.set_state(qpos, qvel)
        return self._get_obs()


To understand what is happening in the environment, you can call `env.render()` to visualize it.  
Since we set `render_mode="rgb_array"`, `env.render()` will return an RGB array that can be displayed.  


> **Note:** If the default view of the environment is not ideal, you may need to adjust the camera.  
> You can do this by defining a camera in your Mujoco XML file and then passing its name as a kwarg to the `TrajEnv` constructor via the `camera_name` argument.


If you are on colab you can render the environment like so. If you are running MuJoCo locally you can use `render_mode="human"` to display viewing window.


In [ ]:
from gymnasium.wrappers import TimeLimit
import matplotlib.pyplot as plt

env = TrajEnv(XML_PATH, 20, render_mode='rgb_array', camera_name="fixed")
env = TimeLimit(env, 100)
env.reset()
image = env.render() 
image = image
plt.imshow(image)

With these core functionalities we can build the basic interaction loop that consists of a reset followed by steps until terminated or truncated is True. We can also record a video as well.

In [ ]:
obs, _ = env.reset()
done = False
images = [env.render()]
# keep taking steps until either we terminated or truncate
while not done:
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    images.append(env.render())
    done = terminated or truncated

In [ ]:
from IPython.display import Video
import imageio

frames = images 
frames = [frame.astype(np.uint8) for frame in frames]   

# Write video to file
video_path = "output_video.mp4"
with imageio.get_writer(video_path, fps=20) as writer:
    for frame in frames:
        writer.append_data(frame)

# Display video in notebook
Video(video_path, embed=True)

We will now try and control the robot to grasp the cube, pick it up, and drop it in the wooden bin. 

The following code sets up a motion planning library for you and saves some useful variables. 
- `planner`: The planner objects uses screw motion planning.
- `panda_hand_tcp`: The panda hand tool center point "link", which gives the current pose of the link we are controlling via the motion planning library. This is the point in the middle between the two panda arm grippers.
- `cube`: The cube object, which you can use to acccess its pose.
- `bin`: The bin object.

The motion planner setup will generate a sequence joint position actions for all the joints excluding the gripper. The gripper actions you can manually pick (can range from 0 to 255). 

In [25]:
import os

# mplib ships Linux wheels only. On macOS the import fails, so the notebook falls
# back to the local IK expert defined in the next cell. Everything else is the same.
try:
    import mplib
except ImportError:
    mplib = None
    print("mplib is unavailable, so planner=None and the local IK expert will be used.\n"
          "That is expected on macOS. Run this notebook in Colab to use the screw planner.")

env = TrajEnv(
    xml_file=XML_PATH, 
    frame_skip=20,
    render_mode="rgb_array"
    
)
env.reset(seed=42)

link_names = [
    'panda_link0','panda_link1','panda_link2','panda_link3','panda_link4',
    'panda_link5','panda_link6','panda_link7','panda_link8','panda_hand',
    'panda_hand_tcp','panda_leftfinger','panda_rightfinger',
    'panda_leftfinger_pad','panda_rightfinger_pad'
]
joint_names = [
    'panda_joint1','panda_joint2','panda_joint3','panda_joint4',
    'panda_joint5','panda_joint6','panda_joint7','panda_finger_joint1','panda_finger_joint2'
]

if mplib is not None:
    planner = mplib.Planner(
        urdf=os.path.abspath("assets/descriptions/panda/urdf/panda.urdf"),
        srdf=os.path.abspath("assets/descriptions/panda/urdf/panda.srdf"),
        user_link_names=link_names,
        user_joint_names=joint_names,
        move_group="panda_hand_tcp",
        joint_vel_limits=np.ones(7) * 0.8,
        joint_acc_limits=np.ones(7) * 0.8,
    )

    # this sets the planner object up such that you can plan with poses in the world frame, which is the default frame of all pose data
    # in our simulator
    planner.set_base_pose(env.base_pose)
else:
    planner = None


**Notes for this task**

The goal is to use the motion planner to pick up the cube and drop it into the bin.

- You will need to access object positions and orientations from the simulator (env.data.xpos, env.data.xquat, etc.).
- The motion planner requires a target pose (7,) for the robot hand (position + quaternion) and the current joint configuration (qpos). Both are expressed in the world frame.
- Plan a trajectory to a consistent grasp pose above the cube, execute it while keeping the gripper open, then move down and close the gripper to pick up the cube.    
- Finally, open the gripper to release the cube into the bin.

The planner may fail if the pose is unreachable, collides with the environment, or the solver does not converge. This is expected—adjust poses or retry if needed.

In [ ]:
from gymnasium.wrappers import RecordVideo
import importlib.util

EPISODES = 10

# RecordVideo needs moviepy, which Colab has and a bare local env may not.
RECORD_VIDEO = importlib.util.find_spec("moviepy") is not None

# Gripper actuator is on a 0-255 scale (see panda_asset.xml actuator8).
GRIPPER_OPEN = 255.0
GRIPPER_CLOSED = 0.0

# One fixed grasp orientation for every episode: the gripper pointing straight
# down. This is the "standardise the approach" tip from the README - the policy
# only has to learn one grasp style instead of many. It is also exactly the TCP
# orientation at the "home" keyframe, so it is always reachable.
TCP_DOWN = np.array([0.0, 1.0, 0.0, 0.0])

# Heights of each phase above the object, in metres.
APPROACH_HEIGHT = 0.12
GRASP_HEIGHT = 0.005     # TCP just above the cube centre, pads around its sides
LIFT_HEIGHT = 0.20
BIN_HEIGHT = 0.16


def ik_delta(base, target_pos, target_quat=TCP_DOWN, iters=200):
    """Joint delta taking the TCP from its CURRENT pose to the target pose.

    Damped least squares on the TCP site Jacobian. This is the fallback expert
    for machines without mplib, and it also provides the small final correction
    described in servo_to_pose below.
    """
    model = base.model
    data = mujoco.MjData(model)
    data.qpos[:] = base.data.qpos
    q_start = data.qpos[:7].copy()

    jacp, jacr = np.zeros((3, model.nv)), np.zeros((3, model.nv))
    err = np.zeros(6)
    for _ in range(iters):
        mujoco.mj_forward(model, data)
        err[:3] = target_pos - data.site_xpos[base.tcp_sid]
        current = np.empty(4)
        mujoco.mju_mat2Quat(current, data.site_xmat[base.tcp_sid])
        conj = np.array([current[0], -current[1], -current[2], -current[3]])
        delta_quat = np.empty(4)
        mujoco.mju_mulQuat(delta_quat, target_quat, conj)
        mujoco.mju_quat2Vel(err[3:], delta_quat, 1.0)
        if np.linalg.norm(err[:3]) < 1e-4 and np.linalg.norm(err[3:]) < 1e-3:
            break
        mujoco.mj_jacSite(model, data, jacp, jacr, base.tcp_sid)
        J = np.vstack([jacp[:, :7], jacr[:, :7]])
        data.qpos[:7] += np.linalg.solve(J.T @ J + 1e-4 * np.eye(7), J.T @ err) * 0.5
        data.qpos[:7] = np.clip(data.qpos[:7], model.jnt_range[:7, 0], model.jnt_range[:7, 1])
    return data.qpos[:7] - q_start


def plan_waypoints(base, setpoint, target_pos, target_quat=TCP_DOWN, n_interp=25):
    """Arm joint waypoints from the current setpoint to the target TCP pose.

    Uses mplib's screw planner when it is available, otherwise interpolates
    towards the IK solution. Returns an (N, 7) array of joint positions.
    """
    if planner is not None:
        target_pose = np.concatenate([target_pos, target_quat])
        result = planner.plan_screw(target_pose, base.data.qpos[:9])
        if str(result.get("status", "")).lower() != "success":
            raise RuntimeError(f"planner failed: {result.get('status')}")
        return np.asarray(result["position"])[:, :7]

    goal = setpoint + ik_delta(base, target_pos, target_quat)
    alphas = np.linspace(0.0, 1.0, n_interp + 1)[1:]
    return setpoint + np.outer(alphas, goal - setpoint)


def record(env, episode_data, action):
    """Step the env, logging the (observation, action) pair for behavior cloning.

    The observation must be the one the expert SAW when it chose this action, so
    it is read before stepping. env may be wrapped (RecordVideo), so step through
    the wrapper but read state off env.unwrapped.
    """
    base = env.unwrapped
    if episode_data is not None:
        episode_data.append((base._get_obs().copy(), np.asarray(action, dtype=np.float32).copy()))
    return env.step(action)


def servo_to_pose(env, setpoint, target_pos, gripper, episode_data,
                  rounds=3, tol=2e-3, settle=8):
    """Drive the TCP to target_pos and return the new commanded setpoint.

    The arm actuators are position servos, so the arm settles slightly short of
    whatever joint angles are commanded (gravity sag): commanding the planner's
    final waypoint leaves 10-50 mm of TCP error, which is enough to miss the cube.
    Each extra round adds the residual back into the setpoint until the measured
    TCP is within tol of the target.
    """
    base = env.unwrapped
    for _ in range(rounds):
        for waypoint in plan_waypoints(base, setpoint, target_pos):
            setpoint = np.asarray(waypoint, dtype=np.float64)
            record(env, episode_data, np.concatenate([setpoint, [gripper]]))
        for _ in range(settle):
            record(env, episode_data, np.concatenate([setpoint, [gripper]]))
        if np.linalg.norm(base.tcp_pos - target_pos) < tol:
            break
    return setpoint


def hold(env, setpoint, gripper, episode_data, steps):
    """Keep the arm still for a number of steps, e.g. while the gripper moves."""
    for _ in range(steps):
        record(env, episode_data, np.concatenate([setpoint, [gripper]]))


def pick_cube_solution(env, episode_data=None):
    """Expert policy: pick the cube up and drop it in the bin.

    Phases: approach -> descend -> close -> lift -> traverse -> release -> settle.
    Appends (obs, action) pairs to episode_data and returns True on success.
    """
    base = env.unwrapped
    setpoint = base.data.qpos[:7].copy()
    cube = base.cube_pos.copy()

    try:
        setpoint = servo_to_pose(env, setpoint, cube + [0, 0, APPROACH_HEIGHT],
                                 GRIPPER_OPEN, episode_data)
        setpoint = servo_to_pose(env, setpoint, cube + [0, 0, GRASP_HEIGHT],
                                 GRIPPER_OPEN, episode_data)
        hold(env, setpoint, GRIPPER_CLOSED, episode_data, 50)
        if not base.is_grasped:
            return False
        setpoint = servo_to_pose(env, setpoint, cube + [0, 0, LIFT_HEIGHT],
                                 GRIPPER_CLOSED, episode_data)
        setpoint = servo_to_pose(env, setpoint, base.bin_pos + [0, 0, BIN_HEIGHT],
                                 GRIPPER_CLOSED, episode_data)
        hold(env, setpoint, GRIPPER_OPEN, episode_data, 30)
        hold(env, setpoint, GRIPPER_OPEN, episode_data, 80)   # let the cube settle
    except RuntimeError as exc:
        # Unreachable pose, collision, or a solver that did not converge. Expected
        # occasionally; the collection loop just skips this episode.
        print(f"  planning failed: {exc}")
        return False

    return base.terminated


RUN_PLANNER_DEMO = True

if RUN_PLANNER_DEMO:
    env = TrajEnv(
        xml_file=XML_PATH,
        frame_skip=20,
        render_mode="rgb_array"
    )
    if RECORD_VIDEO:
        env = RecordVideo(
            env,
            "videos",                         # folder to save videos
            episode_trigger=lambda e: True,    # record every episode
            name_prefix="planner"
        )
    else:
        print("moviepy is not installed, so no video is recorded. `pip install moviepy` to enable it.")

    successes = 0
    for i in range(EPISODES):
        env.reset(seed=i)
        if planner is not None:
            planner.set_base_pose(env.unwrapped.base_pose)
        successes += pick_cube_solution(env)

    print(f"Success rate: {successes / EPISODES}")
    env.close()


In [ ]:
from IPython.display import Video
Video("videos/planner-episode-0.mp4", embed=True, width=320) # Watch replay

## Part 1: Collect your Imitation Learning Dataset


Now that you’ve used the motion planner to complete a single pick-and-place sequence, the next step is to repeat this process many times to collect a dataset for imitation learning.

In imitation learning, the dataset is made of trajectories—sequences of state-action pairs showing how an expert (in this case, the motion planner) solves the task. Each trajectory will later be used to train a policy that imitates this behavior.

>**Tip**: Standardizing the orientation and approach of the gripper will help the robot pick up the cube in the same way each time, resulting in a more reliable model.


In [ ]:
import pickle

# cube_xy_noise is essential here: at the default 0.0 every episode replays the
# identical layout, so the dataset carries no information about how the expert
# reacts to the cube being somewhere else.
env = TrajEnv(
    xml_file=XML_PATH,
    frame_skip=20,
    cube_xy_noise=0.02,
)

trajectories = []

# Start small. Confirm the pickle is non-empty and one trajectory looks sane,
# then raise this to a few hundred. Collection is ~0.3 s/episode with the IK
# backend; mplib's planner is slower, so time 20 before scaling up.
EPISODES = 20
success_count = 0

for i in range(EPISODES):
    env.reset()
    episode_data = []

    # This sets the planner object up such that you can plan with poses in the world frame.
    if planner is not None:
        planner.set_base_pose(env.base_pose)

    result = pick_cube_solution(env, episode_data)

    if result:
        trajectories.append(episode_data)
        print(f"Episode {i} completed successfully.")
        success_count += 1

# ===== Save dataset =====
with open("pick_place_dataset.pkl", "wb") as f:
    pickle.dump(trajectories, f)

print(f"Success rate: {success_count / EPISODES}")
print(f"Total trajectories: {len(trajectories)}")
env.close()


## Part 2: Create Behavior Cloning Dataset

Using the trajectory data you have collected in the previous task, complete the `TrajectoryDataset` class to feed your model. 

>**NOTE**: The model learns to map states to actions, so the order of the data is not important. You can flatten all episodes and sample from any step freely.

In [ ]:
from torch.utils.data import Dataset
import pickle


class TrajectoryDataset(Dataset):
    def __init__(self, data):
        raise NotImplementedError("TODO: flatten trajectory list into (obs, action) training pairs")

    def __len__(self):
        raise NotImplementedError("TODO: return number of flattened samples")

    def __getitem__(self, idx):
        # This should return observation idx and action idx.
        raise NotImplementedError("TODO: return (obs[idx], action[idx])")


dataset_file = "pick_place_dataset.pkl"
with open(dataset_file, "rb") as f:
    data = pickle.load(f)
dataset = TrajectoryDataset(data)


## Part 3: Define the Policy Network

In this step, you will implement a behavior cloning policy network. The goal of this network is to learn to map observations (states) to actions by imitating the trajectories you collected earlier. 

**NOTE**: See [tips](tips.md) for guidance on designing your network architecture and BC overview.

In [ ]:
import torch.nn as nn


class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define the network architecture.
        # Recommended: shared MLP trunk with separate mean/log_std heads.
        raise NotImplementedError("TODO: implement Actor.__init__")

    def forward(self, x):
        raise NotImplementedError("TODO: return mean, log_std")


## Part 4: Training Loop
You will train the neural network policy to imitate expert behavior.

We’ve provided an `eval_policy` function to test your actor in simulation. It runs several episodes and reports a success rate, so you can track how well your policy performs as training progresses.

In [22]:
import gymnasium as gym
import torch.nn as nn
import numpy as np
import torch

def eval_policy(eval_env: gym.Env, actor: nn.Module, num_episodes: int = 5):
    success = 0
    for ep in range(num_episodes):
        obs, _ = eval_env.reset()
        terminated, truncated = False, False
        
        while not (terminated or truncated):
            with torch.no_grad():
                action, _ = actor(torch.from_numpy(obs).float())
            obs, _, terminated, truncated, _ = eval_env.step(action.cpu().numpy())

        if terminated:
            success += 1
        
        print(f"Episode {ep+1}: Success={terminated}")
    return success / num_episodes

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(0)
np.random.seed(0)

actor = Actor().to(device)
eval_env = TrajEnv(XML_PATH, 20)

batch_size = 1024
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

lr = 3e-5
epochs = 100

def nll_loss(actor: nn.Module, obs: torch.Tensor, expert_actions: torch.Tensor):
    mean, log_std = actor(obs)
    std = log_std.exp()
    dist = torch.distributions.Normal(mean, std)
    log_prob = dist.log_prob(expert_actions).sum(-1)   #sum over action dims
    loss = -log_prob.mean()
    return loss

# TODO: Write the training loop
# Train the actor policy to predict the actions in the dataset.
# Make sure to run the eval_policy function every once in a while to track training progress
# and save your best policy checkpoint to be loaded later for evaluation. 
# If you are confident in your model, sometimes a different seed can work. 
# Training time depends on number of samples and number of epochs, but should not take more than 10 minutes on google colab.


## Part 5: Evaluate in Simulator
Time to evaluate your model. You may notice that it doesn’t succeed very often and may behave unpredictably or erratically. 

Can you think about why this might happen?

In [ ]:
import torch
import numpy as np
from gymnasium.wrappers import RecordVideo


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(0)
np.random.seed(0)

eval_env = TrajEnv(XML_PATH, 20, render_mode='rgb_array')
eval_env = RecordVideo(
    eval_env,
    "videos",                       # folder to save videos
    episode_trigger=lambda e: True,  # record every episode
    name_prefix="bc-model"
    )
actor = Actor().to(device)

state_dict = torch.load('best_policy.pth')
actor.load_state_dict(state_dict)

#Evaluation
eval_policy(eval_env, actor, 5)
eval_env.close()

In [ ]:
from IPython.display import Video
Video("videos/bc-model-episode-0.mp4", embed=True, width=320) # Watch our replay